# Batch ODE Kinetic Fitting

Run the full DK → ODE fitting pipeline on all samples in a `.cxw` file (or `.zip` data package).
This notebook demonstrates:
- Loading a `.cxw` experiment file
- Running `batch_fit(mode='ode')` for all samples
- Flagging poor fits and non-specific binders
- Saving individual data-vs-model plots as PNGs
- Exporting a summary CSV

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sensofit import batch_fit, flag_poor_fits
from sensofit.models import is_nonspecific_binder
from sensofit.plotting import plot_fit, save_fit_plots

## Configure Paths

Set the input `.cxw` file (or `.zip` package) and output directory for plots and CSV.

In [ ]:
# --- Reusable loader cell --------------------------------------------------
# `batch_fit` (and the underlying `load_experiment`) accepts:
#   - a `.cxw` file              → uses `sensofit.load_cxw`
#   - a `.zip` data package      → uses `sensofit.load_package`
#   - an unzipped package folder → uses `sensofit.load_package`
INPUT = '../20260318_EV71 2A Binding assay.cxw'
# INPUT = '../20260318_EV71_2A_Binding_assay_pkg.zip'

OUTPUT_DIR = Path('../results/20260318_EV71 2A Binding assay')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Input:  {INPUT}')
print(f'Output: {OUTPUT_DIR.resolve()}')

## Run Batch ODE Fitting

Fits every sample cycle using Direct Kinetics → ODE refinement.
Non-specific binders are detected and skipped automatically.

In [ ]:
df, data = batch_fit(INPUT, mode='dk', include_nsb=True, progress=True)
df = flag_poor_fits(df)
print(f'\n{len(df)} samples fitted, {df["flag"].sum()} flagged')

## Results Summary

Display the kinetic parameters table. Flagged rows indicate fits that may need manual review.

In [ ]:
# Key columns for review
cols = ['compound', 'concentration_uM', 'ka', 'kd', 'KD_uM', 'Rmax',
        'sigma_res', 'nonspecific', 'flag', 'flag_reason']
display_cols = [c for c in cols if c in df.columns]
df[display_cols].head(20)

## Save Fit Plots as PNGs

Re-fit each sample individually to get the full result arrays needed for plotting,
then save annotated data-vs-model overlays.

In [ ]:
from sensofit.ode_fitting import fit_sample as ode_fit_sample

samples = data['samples']

def find_sample(name, conc_uM=None):
    """Find a sample by compound name (substring match), optionally filter by concentration.
    If multiple matches, prompts user to choose."""
    hits = [(i, s) for i, s in enumerate(samples) if name in s['compound']]
    if conc_uM is not None:
        hits = [(i, s) for i, s in hits if abs(s['concentration_M'] * 1e6 - conc_uM) < 0.01]
    if not hits:
        raise ValueError(f'No sample found matching "{name}"' + (f' at {conc_uM} µM' if conc_uM else ''))
    if len(hits) == 1:
        return hits[0][1]
    print(f'Multiple matches for "{name}":')
    for j, (idx, s) in enumerate(hits):
        nsb, _ = is_nonspecific_binder(s)
        tag = ' [NSB]' if nsb else ''
        print(f'  [{j}] samples[{idx}]  {s["compound"]}  {s["concentration_M"]*1e6:.0f} µM  (cycle {s["index"]}){tag}')
    choice = int(input(f'Select [0–{len(hits)-1}]: '))

    return hits[choice][1]print(f'Saved {n_plots} plot(s) → {plot_dir}/')

n_plots = sum(1 for p in paths if p is not None)

results = []paths = save_fit_plots(results, samples, plot_dir, mode='ode')

plot_dir = str(OUTPUT_DIR / 'plots')

for i, sample in enumerate(samples):

    row = df[df['cycle_index'] == sample['index']]        results.append(None)

    if row.empty or row.iloc[0].get('nonspecific', False) or not row.iloc[0].get('success', False):    except Exception:

        results.append(None)        results.append(result)

        continue        result = ode_fit_sample(sample, data['dmso_cals'], blanks=data['blanks'])
    try:

## Example Fit Plot

Display one of the generated plots inline.

In [ ]:
# Show first successful fit
%matplotlib inline
for result, sample in zip(results, samples):
    if result is not None:
        fig = plot_fit(result, sample)
        plt.show()
        break

## Export CSV

Save the full results table with source file, cycle number, sample ID and quality flags.

In [ ]:
csv_path = OUTPUT_DIR / 'batch_ode_results.csv'
df.to_csv(csv_path, index=False)
print(f'Results saved → {csv_path}')
print(f'  {len(df)} total samples')
print(f'  {df["nonspecific"].sum()} non-specific binders')
print(f'  {df["flag"].sum()} flagged fits')